# Aurelia Speed Run — 200 Sim-Years on Colab GPU

This notebook runs the full Aurelia simulation at accelerated speed:
- **1 tick = 1 sim-day** (360 ticks/sim-year)
- **5 worlds × 12,000 NPCs = 60,000 NPCs**
- **No coordinator, no network, no sleep**
- **Snapshots every 10 sim-years**

Expected runtime on Colab GPU: **~3-5 hours for 200 years** at 12K NPCs/world.
For faster results, use `--npcs 3000` (1-1.5 hours for 200 years).

## Architecture
This runs Phase 1-6.6: decision-driven physics → faction engine → escalation ladder → sovereignty pipeline → reconciliation → discovery → federation dynamics → great persons → cross-world movement → policy drift → black swan narrative seeds.

In [ ]:
# ═══ Cell 1: Setup ═══
# Mount Google Drive for persistent output
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tqdm

import sys, os, time, json
from pathlib import Path
from datetime import datetime

In [ ]:
# ═══ Cell 2: Upload & Extract ═══
# Upload aurelia-speed-run.tar.gz to /content/ first
# Then:

!cd /content && tar xzf aurelia-speed-run.tar.gz
!ls /content/aurelia/
!ls /content/aurelia/src_template/*.py | wc -l

sys.path.insert(0, '/content/aurelia')
sys.path.insert(0, '/content/aurelia/src_template')

In [ ]:
# ═══ Cell 3: Run Speed Simulation ═══
# Parameters you can adjust:
SIM_YEARS = 200         # How many sim-years
WORLDS = "solara,valdris,mirithane,arkos,verge"
NPC_COUNT = 12000       # Lower = faster (3000 = ~25% of time)
OUTPUT = "/content/drive/MyDrive/aurelia-output"

!cd /content/aurelia && python3 colab/speed_run.py \
  --years {SIM_YEARS} \
  --worlds {WORLDS} \
  --npcs {NPC_COUNT} \
  --output {OUTPUT} \
  --log-interval 100

In [ ]:
# ═══ Cell 4: Results ═══
import json, glob

snapshots = sorted(glob.glob(f"{OUTPUT}/snapshot_*.json"))
print(f"Found {len(snapshots)} snapshots")

if snapshots:
    with open(snapshots[-1]) as f:
        final = json.load(f)
    print(f"\nFinal state (Year {final['sim_year']}):")
    for world, data in final.get('worlds', {}).items():
        print(f"\n  {world}:")
        print(f"    Population: {data.get('population', 0)}")
        print(f"    Types: {data.get('types', {})}")
        print(f"    Factions active: {data.get('factions_active', 0)}")
        print(f"    Factions at war: {data.get('factions_at_war', 0)}")
        print(f"    Factions integrated: {data.get('factions_integrated', 0)}")
        print(f"    Discoveries: {data.get('discoveries', 0)}")
        print(f"    Great persons: {data.get('great_persons', 0)}")
        print(f"    Active treaties: {data.get('active_treaties', 0)}")
        print(f"    Time: {data.get('world_time', {})}")

    # Timeline
    print(f"\n\nTimeline:")
    for snap in snapshots:
        with open(snap) as f:
            data = json.load(f)
        total_pop = sum(w.get('population', 0) for w in data['worlds'].values())
        total_factions = sum(w.get('factions_active', 0) for w in data['worlds'].values())
        total_war = sum(w.get('factions_at_war', 0) for w in data['worlds'].values())
        print(f"  Year {data['sim_year']:6.1f} | pop:{total_pop:6d} | factions:{total_factions:3d} | at_war:{total_war:2d}")

In [ ]:
# ═══ Cell 5: Download World DBs ═══
# Download individual world databases for local inspection
from google.colab import files
import glob

for db_path in sorted(glob.glob(f"{OUTPUT}/*.db")):
    print(f"Downloading {db_path}...")
    files.download(db_path)
    print(f"  Done: {os.path.basename(db_path)}")